# 5-2. Quantization — 모델 경량화와 추론 최적화

---

## 목차

| # | 내용 |
|:---:|------|
| 0 | **환경 설정** — 패키지 설치, GPU 확인 |
| 1 | **왜 Quantization이 필요한가** — LLM의 메모리 문제와 해결 방향 |
| 2 | **비트 정밀도의 이해** — FP32, FP16, INT8, INT4 비교 |
| 3 | **FP16 모델 로딩과 측정** — 메모리·속도의 기준선 확인 |
| 4 | **INT4 양자화 적용** — bitsandbytes + NF4로 경량화 |
| 5 | **양자화의 부작용** — 환경 변화 입력에서의 품질 저하 |
| 6 | **TTP로 품질 복구** — Test-time Prompting 전략 |
| 7 | **정리** | 5분 |

<br>

> **📌 이 실습은**
>
> 개념을 이해한 뒤, 자기주도 실습(`실습_5-2`)에서 직접 코드를 작성하게 된다.
>
> **⚠️ 참고**: 교육 일정 상 5-1(PEFT/파라미터 효율적 튜닝)보다 5-2(Quantization)를 먼저 진행한다.
> 5-1에서 배울 QLoRA는 "학습 시 양자화"이고, 5-2는 "추론 시 양자화"에 해당한다.
> 5-2를 먼저 이해하면 5-1의 QLoRA가 왜 필요한지 더 잘 이해할 수 있다.


---

## 0. 환경 설정


In [1]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 역할: 양자화 실습에 필요한 패키지 설치
# - transformers: HuggingFace 모델 로딩 및 추론
# - bitsandbytes: INT4/INT8 양자화 지원 (CUDA GPU 필요)
# - accelerate: GPU 자동 분배 (device_map='auto'에 필요)
# - tokenizers: 토크나이저 (transformers 내부 의존성)
# ═══════════════════════════════════════════════════════════
!pip install -q \
    transformers>=4.55.0 bitsandbytes>=0.43.0 \
    torch>=2.0.0 accelerate>=0.30.0 tokenizers>=0.20.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-upstage 0.7.7 requires tokenizers<0.21.0,>=0.20.0, but you have tokenizers 0.22.2 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import time
import warnings
import logging

# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: GPU 환경 확인 + 불필요한 경고 메시지 억제
#
# [Warning 해결]
# 1. triton not found: Windows 환경에서는 triton이 미지원.
#    flop counting(연산량 측정)에만 사용되므로 실습에 영향 없음.
# 2. FutureWarning (_check_is_size): bitsandbytes 내부 코드가
#    PyTorch 차기 버전 API를 아직 반영하지 않은 것. 동작에 영향 없음.
# → 두 경고 모두 실습 결과에 영향을 주지 않으므로 억제한다.
# ═══════════════════════════════════════════════════════════

# 불필요한 경고 억제
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message='.*triton.*')
logging.getLogger('torch.utils.flop_counter').setLevel(logging.ERROR)

# GPU 확인
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name}')
    print(f'VRAM: {gpu.total_memory / 1024**3:.1f} GB')
    print(f'CUDA 버전: {torch.version.cuda}')
else:
    print('⚠️ GPU를 사용할 수 없습니다. CUDA 환경을 확인하세요.')


c:\Users\SSAFY\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⚠️ GPU를 사용할 수 없습니다. CUDA 환경을 확인하세요.


---

## 1. 왜 Quantization이 필요한가?

![image_A](https://i.ibb.co/tP3HgBpG/image-A.png)

### 1-1. LLM의 근본적 문제: 크기

LLM의 성능은 **파라미터 수에 비례**하여 좋아진다.
GPT-3(1,750억)보다 GPT-4(약 1조)가, Llama-3-8B보다 Llama-3-70B가 더 똑똑한 것이 그 증거이다.

하지만 파라미터가 많아질수록 **메모리와 비용**도 비례하여 증가한다.

| 모델 | 파라미터 수 | FP16 메모리 | 필요 GPU |
|------|:---:|:---:|------|
| Qwen2.5-1.5B | 15억 개 | ~3GB | RTX 3060 (12GB) |
| Llama-3-8B | 80억 개 | ~16GB | RTX 4090 (24GB) |
| Llama-3-70B | 700억 개 | ~140GB | A100 × 2장 (160GB) |
| GPT-4급 | ~1조 개 | ~2TB | 수십 장의 고급 GPU |

우리가 사용하는 GPU(RTX 5060ti)의 VRAM은 **16GB**이다.
FP16으로는 8B 모델도 빠듯하고, 13B 이상은 로딩조차 불가능하다.

**문제는 명확하다: 좋은 모델을 쓰고 싶지만, GPU 메모리가 부족하다.**

### 1-2. Quantization(양자화)이란?

**Quantization = 모델의 가중치를 표현하는 비트 수를 줄이는 것**

모델의 가중치(weights)는 숫자이다. 이 숫자를 표현하는 데 사용하는 비트 수를 줄이면
메모리 사용량이 비례하여 감소한다.

> **💡 비유: 사진 압축**
>
> - **FP32** = RAW 사진 (원본 그대로, 파일 크기 큼)
> - **FP16** = JPEG 고화질 (거의 차이 없음, 50% 압축)
> - **INT4** = JPEG 저화질 (약간 흐릿하지만, 87.5% 압축)
>
> 사진 압축처럼, Quantization도 **약간의 품질 손실을 감수하고 크기를 대폭 줄이는 것**이다.
> 핵심은 "눈에 띄지 않을 정도의 손실로 최대한 압축"하는 것이다.

| 정밀도 | 비트 수 | 메모리 (1B 파라미터 기준) | FP32 대비 절감 |
|--------|:---:|:---:|:---:|
| FP32 | 32비트 | 4GB | 기준 |
| FP16 | 16비트 | 2GB | 50% |
| INT8 | 8비트 | 1GB | 75% |
| INT4 | 4비트 | 0.5GB | 87.5% |

INT4 양자화를 적용하면 Llama-3-70B(FP16: 140GB)를 **약 35GB**로 줄일 수 있다.
A100 1장(80GB)에서 충분히 돌릴 수 있게 되는 것이다.

### 1-3. Quantization의 트레이드오프

```
메모리/속도 ↑↑  vs  정확도 ↓
```

비트 수를 줄이면 메모리와 속도가 개선되지만, **정밀도 손실로 인한 품질 저하**가 발생할 수 있다.
특히 **노이즈가 있는 입력**(오타, 모호한 표현)에서 품질 저하가 더 심해진다.

### 1-4. 5-1(PEFT)과의 관계 — 미리보기

| 구분 | 5-2 (이번 챕터) | 5-1 (다음 시간) |
|------|:---:|:---:|
| 시점 | **추론** 시 | **학습** 시 |
| 목적 | 추론 메모리 절감, 속도 향상 | 학습 메모리 절감 (적은 GPU로 튜닝) |
| 방식 | PTQ (학습 완료 후 양자화) | QLoRA (양자화 + 소수 파라미터만 학습) |
| 결과 | 가벼운 추론 모델 | 도메인 특화 모델 |

<br> 

> 5-2에서 "모델을 가볍게 만드는 법"을 배우면,
> 5-1에서 "가볍게 만든 모델을 어떻게 효율적으로 학습시키는지"를 더 잘 이해할 수 있다.


---

## 2. 비트 정밀도의 이해

![image_B](https://i.ibb.co/m5MDYJz9/image-B.png)

### 2-1. 숫자를 비트로 표현하는 방법

컴퓨터는 모든 숫자를 `비트(0과 1)`로 표현한다.
비트 수가 많을수록 더 정밀하게 표현할 수 있지만, 메모리를 더 많이 사용한다.

```
FP32 (32비트): ████████████████████████████████  →  소수점 7~8자리 정밀도
FP16 (16비트): ████████████████                  →  소수점 3~4자리 정밀도
INT8 ( 8비트): ████████                          →  -128 ~ 127 정수 (256단계)
INT4 ( 4비트): ████                              →  -8 ~ 7 정수 (16단계)
```

### 2-2. 부동소수점 vs 정수

| 구분 | FP (Float) | INT (Integer) |
|------|:---:|:---:|
| 표현 방식 | 소수점이 "떠다니는" 실수 | 정수만 표현 |
| 예시 | 0.123456, -3.14 | 0, 3, -7 |
| 장점 | 넓은 범위, 높은 정밀도 | 연산 빠름, 메모리 적음 |
| 용도 | 학습, 원본 가중치 | **양자화된 가중치** |

LLM의 원본 가중치는 FP32/FP16(실수)이지만, 양자화하면 INT8/INT4(정수)로 변환된다.
추론할 때는 이 정수를 다시 실수로 복원(역양자화)하여 연산한다.

> **💡 핵심 질문: 16단계(INT4)만으로 LLM의 가중치를 표현할 수 있는가?**
>
> LLM의 가중치는 대부분 **0 근처에 밀집**된 정규분포를 따른다.
> 극단적으로 크거나 작은 값은 매우 드물다.
> 따라서 0 근처를 더 세밀하게, 극단값은 거칠게 표현해도 품질 손실이 크지 않다.
> 이것이 `NF4(Normal Float 4-bit)`의 핵심 아이디어이다.

### 2-3. 양자화 수식

양자화의 핵심 수식은 다음과 같다:

```
양자화:   Q(x) = round(x / scale) + zero_point
역양자화: x' = (Q(x) - zero_point) × scale
```

| 요소 | 역할 |
|------|------|
| `x` | 원본 가중치 (FP16 실수) |
| `scale` | 실수 범위를 정수 범위에 매핑하는 비율 |
| `zero_point` | 실수 0이 매핑되는 정수 값 |
| `Q(x)` | 양자화된 정수 값 |
| `x'` | 역양자화로 복원된 실수 값 (원본과 약간의 오차) |

양자화 → 역양자화 과정에서 `정보 손실(오차)`이 발생한다.
비트 수가 적을수록 오차가 커지고, 이것이 품질 저하의 원인이 된다.

아래에서 이 과정을 직접 코드로 체험해 보자.


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: FP32 vs FP16의 정밀도 차이를 직접 확인
#
# [왜 이 실험을 하는가?]
# FP32(32비트)와 FP16(16비트)은 같은 숫자를 다른 정밀도로 표현한다.
# FP16은 메모리를 50% 절감하는 대신 소수점 아래 정밀도가 떨어진다.
# 하지만 LLM의 가중치에서 이 정도 오차는 무시할 수 있는 수준이다.
# 그래서 현재 대부분의 LLM 추론은 FP16을 기본 정밀도로 사용한다.
#
# [예상 결과]
# - 원주율(3.14...): FP16도 3.14까지는 정확 → 오차 매우 작음
# - 아주 작은 수(0.00001): FP16에서 약간의 오차 발생
# - FP16 최대값(65504): 정확히 표현 가능 (FP16의 최대 표현 범위)
# - 일반 소수(0.123...): FP16도 0.123까지는 정확
# ═══════════════════════════════════════════════════════════

test_values = [
    3.141592653589793,   # 원주율 — 소수점 몇 자리까지 정확한가?
    0.00001,             # 아주 작은 수 — FP16에서 표현 가능한가?
    65504.0,             # FP16 최대값 근처 — 오버플로우는 없는가?
    0.123456789,         # 일반적인 LLM 가중치 크기 — 실제 사용 범위
]

print('FP32 vs FP16 숫자 표현 비교')
print('=' * 60)
for value in test_values:
    # torch.tensor()로 같은 숫자를 다른 정밀도로 생성
    fp32 = torch.tensor(value, dtype=torch.float32)  # 32비트 실수
    fp16 = torch.tensor(value, dtype=torch.float16)  # 16비트 실수
    diff = abs(fp32.item() - fp16.item())  # 오차 = |FP32 - FP16|
    print(f'원본:  {value}')
    print(f'FP32:  {fp32.item():.15f}')
    print(f'FP16:  {fp16.item():.15f}')
    print(f'오차:  {diff:.15f}')
    print()

# [결과 해석]
# FP16의 오차는 10^-4 ~ 10^-8 수준으로, LLM 가중치 표현에는 충분하다.
# 이것이 FP16이 추론의 기본 정밀도로 사용되는 이유이다.
# 하지만 INT4(16단계)는 이보다 훨씬 거친 표현이므로 오차가 더 커진다.
print('→ FP32와 FP16의 차이는 매우 작다 (10^-4 이하).')
print('  이것이 FP16이 추론의 기본 정밀도로 사용되는 이유이다.')
print('  그렇다면 INT4(16단계)에서는 오차가 얼마나 커질까? → 다음 셀에서 확인')


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: 양자화 수식을 직접 코드로 실행하여 체험
#
# [양자화란?]
# 실수(0.5, -1.2 등)를 정수(3, -2 등)로 변환하는 과정이다.
# 변환 과정에서 "반올림"이 발생하므로 원본과 약간의 오차가 생긴다.
# 비트 수가 적을수록(INT4 < INT8) 표현 가능한 단계가 줄어 오차가 커진다.
#
# [수식]
# 양자화:   Q(x) = round(x / scale) + zero_point
# 역양자화: x' = (Q(x) - zero_point) × scale
#
# scale = 데이터 범위를 정수 범위에 매핑하는 "축척 비율"
# zero_point = 실수 0이 매핑되는 정수 값
#
# [예상 결과]
# INT8(256단계): 오차가 매우 작음 (거의 원본과 동일)
# INT4(16단계): 오차가 INT8의 약 10~20배 → 이것이 품질 저하의 원인
# ═══════════════════════════════════════════════════════════

# 1. 원본 텐서: LLM 가중치 5개를 시뮬레이션
# 실제 LLM은 수십억 개의 가중치를 갖지만, 원리는 동일하다.
original = torch.tensor([0.5, 1.2, -0.3, 2.1, -1.5], dtype=torch.float16)
print(f'1. 원본 텐서 (FP16): {original.tolist()}')

# 2. 데이터 범위 분석: 양자화의 scale을 계산하기 위한 준비
# scale = (최대값 - 최소값) / (표현 가능한 단계 수)
x_min, x_max = original.min().item(), original.max().item()
print(f'2. 데이터 범위: [{x_min:.4f}, {x_max:.4f}]')

# ========== INT8 양자화 (8비트 = 256단계) ==========
# 256단계로 나누므로 scale이 매우 작다 → 정밀한 표현 가능
levels_int8 = 2**8 - 1  # 255 (0~255까지 256개 값)
scale_int8 = (x_max - x_min) / levels_int8  # 각 단계의 간격
zero_point_int8 = round(-x_min / scale_int8) # 0이 매핑되는 정수

# ★ 핵심 수식: Q(x) = round(x / scale) + zero_point
quantized_int8 = torch.round(original / scale_int8) + zero_point_int8
# 역양자화: 정수를 다시 실수로 복원 (이때 오차가 발생한다)
dequantized_int8 = (quantized_int8 - zero_point_int8) * scale_int8

print(f'\n3. INT8 양자화 (256단계):')
print(f'   scale = {scale_int8:.6f} (각 단계의 간격)')
print(f'   양자화 값:  {quantized_int8.tolist()} ← 정수로 변환됨')
print(f'   복원 값:    {[round(v, 4) for v in dequantized_int8.tolist()]} ← 원본과 비교')
error_int8 = (original.float() - dequantized_int8.float()).abs().mean().item()
print(f'   평균 오차:  {error_int8:.6f} ← 256단계이므로 오차가 매우 작다')

# ========== INT4 양자화 (4비트 = 16단계) ==========
# 16단계로 나누므로 scale이 크다 → 거친 표현, 더 큰 오차
levels_int4 = 2**4 - 1  # 15 (0~15까지 16개 값)
scale_int4 = (x_max - x_min) / levels_int4  # INT8보다 ~17배 큰 간격
zero_point_int4 = round(-x_min / scale_int4)

quantized_int4 = torch.round(original / scale_int4) + zero_point_int4
quantized_int4 = quantized_int4.clamp(0, levels_int4)  # 0~15 범위로 제한
dequantized_int4 = (quantized_int4 - zero_point_int4) * scale_int4

print(f'\n4. INT4 양자화 (16단계):')
print(f'   scale = {scale_int4:.6f} (INT8의 {scale_int4/scale_int8:.0f}배 → 더 거친 표현)')
print(f'   양자화 값:  {quantized_int4.tolist()} ← 0~15 사이 정수')
print(f'   복원 값:    {[round(v, 4) for v in dequantized_int4.tolist()]} ← 원본과 차이 발생')
error_int4 = (original.float() - dequantized_int4.float()).abs().mean().item()
print(f'   평균 오차:  {error_int4:.6f} ← INT8보다 {error_int4/max(error_int8,1e-10):.0f}배 큰 오차')

# [결과 해석]
# INT4의 오차가 INT8보다 크지만, 이 정도는 LLM에서 허용 가능한 수준이다.
# 다만 이 "작은 오차"가 수십억 개 파라미터에 걸쳐 누적되면
# 특히 노이즈/오타 같은 비정상 입력에서 품질 저하로 이어진다. (챕터 5)
print(f'\n→ INT4 오차가 INT8보다 크지만, 단일 가중치 수준에서는 허용 범위이다.')
print(f'  문제는 이 오차가 15억 개 파라미터에 걸쳐 "누적"된다는 것이다. (챕터 5에서 확인)')


---

## 3. FP16 모델 로딩과 측정 — 기준선 확인

### 3-1. 왜 기준선(Baseline)이 필요한가?

양자화의 효과를 제대로 평가하려면 **비교 대상**이 필요하다.
"INT4 모델의 메모리가 1.7GB"라고만 해서는 좋은 건지 알 수 없다.
"FP16이 3.9GB인데 INT4가 1.7GB"라고 해야 **56% 절감**이라는 의미가 전달된다.

이 챕터에서 측정할 기준선:

| 지표 | 설명 | 왜 중요한가 |
|------|------|------|
| **모델 메모리** | 가중치가 차지하는 GPU 메모리 | 양자화의 직접적 절감 효과 |
| **피크 메모리** | 추론 중 최대 GPU 메모리 | 실제 운영에서의 VRAM 한도 |
| **추론 시간 (Latency)** | 응답 생성까지 걸리는 시간 | 서비스 응답 속도 |
| **응답 품질** | 생성된 텍스트의 정확성 | 양자화로 인한 품질 손실 여부 |

### 3-2. 이 실습에서 사용하는 모델

**`Qwen/Qwen2.5-1.5B-Instruct`**

| 항목 | 값 |
|------|------|
| 파라미터 수 | 약 15억 개 |
| FP16 예상 메모리 | 약 3~4GB |
| 선택 이유 | 16GB VRAM에서 FP16/INT4 **모두** 로딩 가능, 비교 실험에 적합 |

### 3-3. HuggingFace에서 모델 로딩하기

HuggingFace의 `AutoModelForCausalLM.from_pretrained()`로 모델을 로딩한다.

핵심 파라미터:

```python
model = AutoModelForCausalLM.from_pretrained(
    'Qwen/Qwen2.5-1.5B-Instruct',
    dtype=torch.float16,  # 정밀도 지정 (FP16)
    device_map='auto',          # GPU에 자동 배치
)
```

- `dtype`: 모델 가중치의 정밀도. `torch.float16`이면 FP16으로 로딩
- `device_map='auto'`: GPU 메모리를 고려하여 자동 배치 (`accelerate` 라이브러리 필요)


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: FP16 모델 로딩 + 모델 메모리 측정
#
# [핵심 파라미터]
# - dtype=torch.float16: 가중치를 16비트 부동소수점으로 로딩
#   (32비트 대비 메모리 50% 절감, 품질 차이 거의 없음)
# - device_map='auto': GPU 메모리를 고려하여 자동 배치
#   (accelerate 라이브러리가 내부적으로 처리)
#
# [메모리 측정 방법]
# torch.cuda.memory_allocated(): 현재 GPU에 할당된 메모리
# → 모델 로딩 직후 측정하면 = 가중치(weights)만의 메모리
# ═══════════════════════════════════════════════════════════

model_name = 'Qwen/Qwen2.5-1.5B-Instruct'

print('FP16 모델 로딩 중...')
torch.cuda.reset_peak_memory_stats()  # 메모리 통계 초기화

# ★ 핵심: dtype=torch.float16으로 16비트 정밀도 지정
model_fp16 = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,  # FP16 정밀도
    device_map='auto',          # GPU에 자동 배치
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 모델 로딩 직후 메모리 = 가중치만의 크기
model_memory_fp16 = torch.cuda.memory_allocated() / 1024**3
print(f'모델 로딩 완료: {model_name}')
print(f'FP16 모델 메모리: {model_memory_fp16:.2f} GB')

# [예상 결과]
# Qwen2.5-1.5B = 약 15억 파라미터 × 2바이트(FP16) ≈ 3GB
# 실제로는 임베딩 테이블 등의 오버헤드로 ~3.9GB 정도 나온다.


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: 파라미터 수 → 이론적 메모리 계산
#
# [계산 공식]
# 메모리(GB) = 파라미터 수 × 바이트/파라미터 ÷ (1024^3)
# - FP32: 1개 파라미터 = 4바이트 (32비트 ÷ 8)
# - FP16: 1개 파라미터 = 2바이트 (16비트 ÷ 8)
# - INT8: 1개 파라미터 = 1바이트 ( 8비트 ÷ 8)
# - INT4: 1개 파라미터 = 0.5바이트 (4비트 ÷ 8)
#
# [왜 이론값과 실측값이 다른가?]
# 이론값은 "순수 가중치"만 계산한다.
# 실측값에는 모델 구조 메타데이터, 임베딩 테이블, 버퍼 등
# 추가 오버헤드가 포함되어 이론값보다 약간 크게 나온다.
# ═══════════════════════════════════════════════════════════

# p.numel(): 하나의 파라미터 텐서에 포함된 숫자(원소)의 개수
total_params = sum(p.numel() for p in model_fp16.parameters())

# 정밀도별 이론적 메모리 계산
memory_fp32 = total_params * 4 / 1024**3   # 32비트 = 4바이트/파라미터
memory_fp16_theory = total_params * 2 / 1024**3   # 16비트 = 2바이트
memory_int8 = total_params * 1 / 1024**3   # 8비트 = 1바이트
memory_int4 = total_params * 0.5 / 1024**3 # 4비트 = 0.5바이트

print(f'총 파라미터 수: {total_params:,}개 ({total_params/1e9:.2f}B)')
print(f'\n정밀도별 이론적 메모리 (순수 가중치만):')
print(f'  FP32: {memory_fp32:.2f} GB')
print(f'  FP16: {memory_fp16_theory:.2f} GB (FP32 대비 50% 절감)')
print(f'  INT8: {memory_int8:.2f} GB (FP32 대비 75% 절감)')
print(f'  INT4: {memory_int4:.2f} GB (FP32 대비 87.5% 절감)')

print(f'\n실측값: {model_memory_fp16:.2f} GB (이론값 {memory_fp16_theory:.2f}GB + 오버헤드)')
print(f'→ 같은 모델이 INT4에서는 FP16의 약 1/4 메모리만 사용한다.')


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: 추론 측정 함수 정의 + FP16 기준선 추론
#
# [measure_inference 함수]
# 모델에 질문을 보내고 3가지를 측정한다:
# 1. response: 생성된 텍스트 (응답 품질 확인용)
# 2. latency: 추론에 걸린 시간 (초)
# 3. peak_memory: 추론 중 최대 GPU 메모리 (GB)
#
# [apply_chat_template]
# Qwen 같은 Instruct 모델은 특정 대화 형식을 기대한다.
# apply_chat_template()이 이 형식으로 자동 변환해 준다.
# 예: {role: 'user', content: '질문'} → '<|im_start|>user\n질문<|im_end|>'
#
# [do_sample=False]
# 랜덤 샘플링을 끄고 항상 가장 확률 높은 토큰을 선택한다.
# FP16과 INT4 비교 시 "같은 조건"을 보장하기 위한 설정이다.
# ═══════════════════════════════════════════════════════════

def measure_inference(model, tokenizer, prompt, max_new_tokens=128):
    """모델 추론을 수행하고 latency와 peak memory를 측정한다."""
    torch.cuda.synchronize()  # GPU 작업 완료 대기
    torch.cuda.reset_peak_memory_stats()  # 피크 메모리 통계 초기화

    # 입력 준비: chat template 적용
    messages = [{'role': 'user', 'content': prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors='pt',
        return_dict=True, add_generation_prompt=True
    ).to(model.device)

    # 추론 실행 + 시간 측정
    start = time.time()
    with torch.no_grad():  # 그래디언트 계산 비활성화 (추론 시 불필요)
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False  # 결정적 출력 (비교 일관성)
        )
    torch.cuda.synchronize()  # GPU 작업 완료 대기
    latency = time.time() - start

    # 응답 디코딩: 입력 부분을 제외하고 새로 생성된 토큰만 추출
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],  # [입력길이:] = 새 토큰만
        skip_special_tokens=True
    )

    # 피크 메모리: 추론 과정에서 GPU 메모리가 가장 높았던 순간의 값
    # (가중치 + 활성화값 + KV 캐시 + 임시 버퍼 포함)
    peak_memory = torch.cuda.max_memory_allocated() / 1024**3
    return response, latency, peak_memory

# ========== FP16 기준선 추론 ==========
test_prompt = '서울에서 부산까지 KTX로 얼마나 걸리나요?'
response_fp16, latency_fp16, peak_fp16 = measure_inference(
    model_fp16, tokenizer, test_prompt
)

print(f'질문: {test_prompt}')
print(f'응답: {response_fp16[:200]}')
print(f'\n추론 시간: {latency_fp16:.2f}초')
print(f'모델 메모리: {model_memory_fp16:.2f} GB (가중치만)')
print(f'피크 메모리: {peak_fp16:.2f} GB (추론 중 최대 = 가중치 + 활성화 + KV캐시)')

# [결과 해석]
# 모델 메모리 < 피크 메모리: 추론 중 활성화값과 KV 캐시가 추가로 생성되기 때문
# 1.5B 모델에서는 이 차이가 작지만, 7B+ 모델에서는 차이가 커진다.


---

## 4. INT4 양자화 적용 — bitsandbytes + NF4

![image_C](https://i.ibb.co/d03m8fXF/image-C.png)

### 4-1. PTQ (Post-Training Quantization)

양자화를 적용하는 시점에 따라 크게 두 가지 방식이 있다.

| 구분 | PTQ (이번 실습) | QAT (참고) |
|------|:---:|:---:|
| 이름 | Post-Training Quantization | Quantization-Aware Training |
| 시점 | **학습 완료 후** | 학습 중 |
| 원리 | 완성된 모델의 가중치를 정수로 변환 | 학습 과정에서 양자화 오차를 함께 학습 |
| 재학습 | **불필요** | 필요 |
| 적용 난이도 | **낮음** (코드 1~2줄) | 높음 (학습 파이프라인 수정) |
| 품질 | 약간 손실 가능 | 손실 최소화 |

<br>

이 실습에서는 **PTQ**를 사용한다. 이미 학습된 Qwen2.5-1.5B 모델에 양자화를 "사후 적용"하는 것이다.
코드 한 줄(`quantization_config=...`)만 추가하면 되므로 가장 간편하다.

### 4-2. NF4 (Normal Float 4-bit) — LLM에 최적화된 양자화

일반 INT4는 -8~7의 16단계를 **균등하게** 배분한다.

하지만 LLM의 가중치는 0 근처에 밀집된 **정규분포**를 따른다.

```
[일반 INT4]        균등 분포 (모든 구간 동일 간격)
  ├──┼──┼──┼──┼──┼──┼──┼──┼──┼──┼──┼──┼──┼──┤
 -8                    0                     7

[NF4]              정규분포 최적화 (0 근처 더 세밀)
  ├┼┼┼┼──┼──┼──┼──┼──┼──┼──┼──┤┼┼┼┼┤
 -8           0 (촘촘)              7
```

NF4는 0 근처 구간을 더 조밀하게 배분하여 LLM 가중치의 분포에 최적화되어 있다.
일반 INT4보다 **같은 4비트에서도 품질 손실이 적다**.

### 4-3. BitsAndBytesConfig — 핵심 파라미터

bitsandbytes는 HuggingFace와 연동되는 양자화 라이브러리이다.
`BitsAndBytesConfig`로 양자화 옵션을 설정한다.

```python
BitsAndBytesConfig(
    load_in_4bit=True,                    # INT4 양자화 활성화
    bnb_4bit_compute_dtype=torch.float16, # 연산은 FP16으로 수행
    bnb_4bit_quant_type='nf4',            # NF4 양자화 타입
    bnb_4bit_use_double_quant=True,       # 이중 양자화 (추가 절감)
)
```

| 파라미터 | 역할 |
|---------|------|
| `load_in_4bit` | INT4 양자화 활성화 |
| `bnb_4bit_compute_dtype` | 양자화된 가중치를 **연산할 때** 사용하는 dtype. FP16 권장 |
| `bnb_4bit_quant_type` | `'nf4'`(정규분포 최적화) 또는 `'fp4'` |
| `bnb_4bit_use_double_quant` | 양자화 상수(scale)를 한 번 더 양자화하여 추가 메모리 절감 |

<br>

> **💡 `bnb_4bit_compute_dtype`이 중요한 이유**
>
> 가중치는 INT4로 **저장**되지만, 행렬 곱셈 등 실제 **연산**은 FP16으로 수행된다.
> 즉, 추론 시 INT4 → FP16으로 역양자화한 뒤 연산하는 것이다.
> 저장은 작게, 연산은 정밀하게 — 이것이 bitsandbytes의 핵심 전략이다.

### 4-4. 실무에서의 양자화 방식 선택

| 방식 | 특징 | 적합한 상황 |
|------|------|------|
| **bitsandbytes** | 런타임 양자화, HuggingFace 연동 | 빠른 실험, 프로토타이핑 (이 실습) |
| **GPTQ** | 사전 양자화, 가중치 보정 | 배포 최적화 |
| **AWQ** | 활성화 기반, 중요 가중치 보존 | 품질 중시 서비스 |
| **GGUF** | llama.cpp 형식, CPU 지원 | 로컬/엣지 추론 |


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: INT4 양자화 모델 로딩 + FP16과의 메모리 비교
#
# [중요: FP16 모델을 먼저 GPU에서 해제한다]
# GPU 메모리에 FP16 모델이 남아 있는 상태에서 INT4를 로딩하면
# 두 모델이 동시에 GPU에 존재하여 메모리가 합산된다.
# 정확한 INT4 단독 메모리를 측정하려면 FP16을 먼저 해제해야 한다.
#
# [BitsAndBytesConfig 파라미터 복습]
# - load_in_4bit: INT4 양자화 활성화
# - bnb_4bit_compute_dtype: 연산 시 사용하는 dtype (FP16 권장)
#   → 가중치는 INT4로 "저장"하되, 행렬곱 등 연산은 FP16으로 수행
# - bnb_4bit_quant_type: 'nf4' = 정규분포 최적화 양자화
# - bnb_4bit_use_double_quant: 양자화 상수를 한 번 더 양자화 (추가 절감)
# ═══════════════════════════════════════════════════════════

# ========== FP16 모델 결과 저장 후 GPU에서 해제 ==========
# 비교를 위해 FP16 결과를 딕셔너리에 저장해 둔다.
fp16_results = {
    'model_memory': model_memory_fp16,
    'peak_memory': peak_fp16,
    'latency': latency_fp16,
    'response': response_fp16,
}

# ★ 핵심: GPU 메모리에서 FP16 모델을 완전히 해제
del model_fp16
torch.cuda.empty_cache()  # GPU 캐시까지 비워야 메모리가 실제로 반환됨
import gc; gc.collect()    # Python 가비지 컬렉터도 실행

print(f'FP16 모델 해제 후 GPU 메모리: {torch.cuda.memory_allocated() / 1024**3:.2f} GB')

# ========== INT4 양자화 모델 로딩 ==========
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,                          # INT4 양자화
    bnb_4bit_compute_dtype=torch.float16,       # 연산은 FP16
    bnb_4bit_quant_type='nf4',                  # NF4 양자화
    bnb_4bit_use_double_quant=True,             # 이중 양자화
)

print('\nINT4 양자화 모델 로딩 중...')
torch.cuda.reset_peak_memory_stats()

# ★ 핵심: quantization_config를 from_pretrained에 전달
model_int4 = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map='auto',
)

# INT4 모델 단독 메모리 (FP16이 해제된 상태이므로 정확한 값)
model_memory_int4 = torch.cuda.memory_allocated() / 1024**3
print(f'\nINT4 모델 메모리: {model_memory_int4:.2f} GB')
print(f'FP16 모델 메모리: {fp16_results["model_memory"]:.2f} GB (저장된 값)')
print(f'메모리 절감: {(1 - model_memory_int4/fp16_results["model_memory"])*100:.1f}%')

# [예상 결과]
# FP16: ~3.9GB → INT4: ~1.7GB → 약 56% 절감
# 이론적으로는 75% 절감이지만, 양자화 상수(scale, zero_point) 저장과
# 일부 레이어(임베딩 등)가 FP16으로 유지되어 실제 절감률은 이론보다 낮다.


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: FP16 vs INT4 전체 성능 비교
#
# [비교 지표 3가지]
# 1. 모델 메모리: 가중치만의 크기 → INT4가 확실히 작다
# 2. 피크 메모리: 추론 중 최대값 → 활성화값 때문에 차이가 줄어든다
# 3. 추론 시간: 양자화가 항상 빨라지는 것은 아니다
#    (역양자화 오버헤드가 있어 1.5B 소형 모델에서는 비슷하거나 느릴 수 있음)
# ═══════════════════════════════════════════════════════════

# INT4 추론 실행 (같은 질문으로 비교)
response_int4, latency_int4, peak_int4 = measure_inference(
    model_int4, tokenizer, test_prompt
)

# 저장된 FP16 결과와 비교
fp16_mm = fp16_results['model_memory']
fp16_pk = fp16_results['peak_memory']
fp16_lat = fp16_results['latency']

print('FP16 vs INT4 비교')
print('=' * 70)
print(f'{"지표":<20} {"FP16":>15} {"INT4":>15} {"변화":>15}')
print('-' * 70)
print(f'{"모델 메모리":<20} {fp16_mm:>14.2f}GB {model_memory_int4:>14.2f}GB {(1-model_memory_int4/fp16_mm)*100:>13.1f}%↓')
print(f'{"피크 메모리":<20} {fp16_pk:>14.2f}GB {peak_int4:>14.2f}GB {(1-peak_int4/fp16_pk)*100:>13.1f}%↓')
print(f'{"추론 시간":<20} {fp16_lat:>14.2f}초 {latency_int4:>14.2f}초')
print('=' * 70)

print(f'\nFP16 응답: {fp16_results["response"][:200]}')
print(f'\nINT4 응답: {response_int4[:200]}')

# [결과 해석]
# 모델 메모리: INT4가 확실히 작다 (약 50~60% 절감)
# 피크 메모리: 절감률이 모델 메모리보다 낮다
#   → 이유: 양자화는 가중치만 압축. 활성화값/KV캐시는 FP16 그대로
# 추론 시간: 1.5B 소형 모델에서는 INT4가 반드시 빠르지 않을 수 있다
#   → 이유: INT4→FP16 역양자화 오버헤드가 있기 때문
#   → 7B+ 대형 모델에서는 메모리 절감 효과가 속도 향상으로 이어진다


### 4-5. 모델 메모리 vs 피크 메모리 — 왜 다른가?

비교 결과에서 **모델 메모리 절감은 크지만, 피크 메모리 절감은 작을 수 있다**.
이 차이를 이해하는 것이 양자화 효과를 올바르게 평가하는 데 중요하다.

| 지표 | 측정 시점 | 포함 내용 |
|------|---------|----------|
| **모델 메모리** | 로딩 직후 | 가중치(weights, biases)만 |
| **피크 메모리** | 추론 중 최대 | 가중치 + **활성화값 + KV 캐시 + 임시 버퍼** |

**핵심**: 양자화는 **가중치만 압축**한다. 추론 중 생성되는 활성화값(activations)과
KV 캐시(attention의 Key-Value 저장소)는 여전히 FP16으로 유지된다.
따라서 피크 메모리 절감 효과는 모델 메모리 절감보다 항상 작다.

```
FP16:  모델 ~3.9GB → 피크 ~4.0GB  (차이 ~0.1GB: 활성화 비중 작음)
INT4:  모델 ~1.7GB → 피크 ~3.8GB  (차이 ~2.1GB: 활성화 비중 큼!)
```

> **💡 모델이 클수록 양자화 효과가 극적이다**
>
> 1.5B 모델은 전체 메모리에서 가중치 비중이 상대적으로 작아 효과가 제한적이다.
> 하지만 7B, 13B, 70B 모델에서는 가중치가 전체 메모리의 대부분을 차지하므로
> INT4 양자화 시 **피크 메모리도 극적으로 줄어든다**.
> 실무에서 양자화의 진가는 **큰 모델에서** 드러난다.


---

## 5. 양자화의 부작용 — 환경 변화 입력에서의 품질 저하

![image_D](https://i.ibb.co/WWGsgycX/image-D.png)

### 5-1. 환경 변화(Distribution Shift)란?

LLM은 방대한 텍스트 데이터로 학습된다. 학습 데이터에는 문법적으로 정확하고
명확한 문장이 대부분이다. 하지만 **실제 서비스에서 사용자 입력은 학습 데이터와 다를 수 있다.**

이처럼 학습 데이터와 실제 입력의 **분포가 달라지는 현상**을 `환경 변화(Distribution Shift)`라 한다.

| 유형 | 예시 | 특징 |
|------|------|------|
| **정상** | "서울에서 부산까지 KTX로 얼마나 걸리나요?" | 문법 정확, 맥락 명확 |
| **오타** | "서울에**셔** 부산까지 KTX로 얼마나 걸리나요?" | 학습 데이터에 없는 오타 포함 |
| **노이즈** | "ktx 서울 부산 몇시간? 걍 대충 알려줘 ㅋㅋ" | 불필요한 표현, 구어체 |
| **모호함** | "그거 얼마나 걸려?" | 주어/목적어 생략, 맥락 부족 |
| **조건 추론** | "오후 3시에 서울역에서 KTX 타면 부산에 몇 시에 도착해?" | 복잡한 조건 추가 |

### 5-2. 왜 양자화 모델이 환경 변화에 더 취약한가?

FP16 모델도 환경 변화에 완벽하지 않지만, **양자화 모델은 더 심하게** 품질이 떨어진다.
세 가지 근본 원인이 있다.

#### ① 정밀도 손실 누적

양자화는 각 가중치에서 **작은 오차**를 발생시킨다.
이 오차가 수십억 개의 파라미터 전체에 걸쳐 누적되면 출력에 영향을 미친다.

```
FP16 가중치:  [0.123456, 0.234567, 0.345678, ...]
                  ↓ 양자화
INT4 가중치:  [0.125000, 0.234375, 0.343750, ...]  ← 각각 작은 오차
                  ↓ 수십억 번 연산
최종 출력: 오차가 누적되어 다른 결과 가능
```

#### ② Attention Score 변화

LLM의 핵심인 Self-Attention은 토큰 간 관계를 계산한다.
양자화 오차로 attention score가 미세하게 달라지면, **다른 토큰에 집중**하게 된다.

```
입력: "서울에셔 부산까지" (오타 포함)

FP16:  "에셔"를 "에서"와 유사하게 인식 (높은 attention)
INT4:  미세한 차이로 "에셔"를 다르게 인식 (낮은 attention) → 잘못된 해석
```

#### ③ 경계 케이스(Edge Case) 처리 능력 저하

| 입력 패턴 | FP16 | INT4 |
|---------|:---:|:---:|
| 자주 본 패턴 ("서울에서 부산까지") | ✅ 강함 | ✅ 강함 |
| 드문 패턴 (오타, 모호함) | ✅ 적당히 처리 | ❌ 취약 |

자주 본 패턴은 양자화 후에도 강하게 유지되지만,
`드물게 본 패턴(경계 케이스)`은 정밀도 손실의 영향을 더 크게 받는다.

아래에서 이 현상을 직접 확인해 보자.

### 5-3. 실습에서 무엇을 관찰해야 하는가?

양자화 모델의 품질 저하는 `"답변을 못 하는 것"`이 아니라 `"미묘하게 이상한 답변을 하는 것"`이다.
최신 모델(Qwen2.5 등)은 단순 오타 정도는 잘 처리하므로, 다음과 같은 **미묘한 이상 징후**를 관찰해야 한다.

| 이상 징후 | 설명 | 서비스 영향 |
|---------|------|------|
| **외국어 혼입** | 한국어 답변에 중국어/일본어 문자 등장 (예: 高速鉄路) | 사용자 혼란, 신뢰도 하락 |
| **없는 정보 생성** | 존재하지 않는 노선, 서비스, 요금을 지어냄 | 잘못된 안내, 클레임 |
| **맥락 이탈** | 질문 주제를 벗어난 답변 (KTX → 항공편) | 답변 품질 저하 |
| **답변 불안정** | 중간 끊김, 반복, 비일관적 형식 | 전문성 의심 |

<br>

> **💡 "정답을 맞히는 것"과 "서비스 품질"은 다르다**
>
> "약 2시간 30분"이라는 핵심 정보를 맞히더라도, 답변에 중국어가 섞이거나
> 없는 서비스(KTX E)를 언급하면 고객 서비스로서는 실격이다.
> TTP는 이런 **미묘한 품질 문제**를 보완하는 전략이다.


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: 환경 변화 입력에서 INT4 모델의 품질 저하 관찰
#
# [환경 변화(Distribution Shift)란?]
# 학습 데이터에는 문법적으로 정확한 문장이 대부분이다.
# 하지만 실제 사용자 입력에는 오타, 구어체, 맥락 부족 등이 포함된다.
# 이런 입력이 학습 데이터의 "분포"와 다르므로 "환경 변화"라 부른다.
#
# [관찰 가이드 — 무엇을 봐야 하는가?]
# 양자화 모델은 "답변을 아예 못 하는" 것이 아니라
# "미묘하게 이상한 답변"을 하는 경우가 많다.
# 아래 체크리스트를 기준으로 관찰하라:
#   □ 한국어 질문에 중국어/일본어 문자가 섞여 나오는가? (예: 高速鉄路)
#   □ 존재하지 않는 정보를 지어내는가? (예: KTX E, 350km/h 등)
#   □ 질문의 맥락을 벗어나는가? (예: KTX를 물었는데 항공사를 답변)
#   □ 답변이 중간에 끊기거나 반복되는가?
#   □ "정확한 듯 보이지만 틀린 정보"가 포함되어 있는가?
#
# 이런 미묘한 품질 저하가 실제 서비스에서는 심각한 문제가 된다.
# 고객이 받는 답변에 중국어가 섞이거나, 없는 서비스를 안내하면
# 신뢰도가 급격히 하락하기 때문이다.
# ═══════════════════════════════════════════════════════════

test_samples = [
    # 정상: 기준선 (이 결과와 나머지를 비교)
    {'type': '정상',
     'prompt': '서울에서 부산까지 KTX로 얼마나 걸리나요?'},

    # 오타: 여러 글자에 오타를 섞어 난이도를 높임
    {'type': '오타',
     'prompt': '서울에셔 부산까지 KTX로 얼마나 걸리나요?'},

    # 구어체 + 줄임말: 실제 채팅에서 흔한 형태
    {'type': '구어체',
     'prompt': 'ktx 서울 부산 몇시간? 걍 대충 알려줘 ㅋㅋ'},

    # 맥락 부족 + 복합 질문: 양자화 모델이 추론 여유 부족으로 어려워하는 유형
    {'type': '복합질문',
     'prompt': '서울에서 부산 가는데 KTX랑 SRT 중에 뭐가 더 빠르고 싼지 비교해줘'},

    # 조건 질문 (산술 없음): 상황 판단이 필요한 질문
    {'type': '조건판단',
     'prompt': '주말에 서울역에서 KTX 자유석 타면 앉을 수 있을까요?'},

    # 숫자/조건 포함: 정확한 산술 추론이 필요한 질문
    # ※ 양자화 모델은 시간 계산 같은 산술 추론에서 특히 오류가 크다.
    #    이 유형은 TTP로도 교정이 어렵다 (TTP는 방향 제시이지, 계산 교정이 아님)
    {'type': '산술추론',
     'prompt': '오후 3시에 서울역에서 KTX 타면 부산에 몇 시에 도착해?'},
]

print('INT4 모델 — 환경 변화 입력 테스트')
print('=' * 70)

env_results = []
for sample in test_samples:
    response, latency, _ = measure_inference(
        model_int4, tokenizer, sample['prompt'], max_new_tokens=150
    )
    env_results.append({'type': sample['type'], 'response': response})
    print(f'\n[{sample["type"]}]')
    print(f'입력: {sample["prompt"]}')
    print(f'응답: {response[:300]}')

print('\n' + '=' * 70)
print()
print('📋 관찰 체크리스트 — 아래 항목이 발견되는지 확인하라:')
print('  □ 한국어 답변에 중국어/일본어 문자가 섞여 있는가? (예: 高速鉄路, 铁道)')
print('  □ 존재하지 않는 정보를 생성했는가? (예: 없는 노선, 틀린 요금)')
print('  □ 질문 맥락을 벗어난 답변이 있는가? (예: KTX를 물었는데 비행기 설명)')
print('  □ 답변이 중간에 끊기거나 같은 말을 반복하는가?')
print('  □ 구어체/줄임말을 이해하지 못하고 엉뚱한 해석을 했는가?')
print()
print('⚠️ 참고: 최신 모델(Qwen2.5)은 단순 오타 정도는 잘 처리할 수 있다.')
print('  하지만 "미묘하게 이상한 답변" (중국어 혼입, 없는 정보 생성 등)은')
print('  실제 고객 서비스에서 신뢰도를 크게 떨어뜨리는 문제이다.')
print('  TTP는 이런 미묘한 품질 저하를 보완하는 전략이다.')


---

## 6. TTP(Test-time Prompting)로 품질 복구

![image_E](https://i.ibb.co/xnVVwVF/image-E.png)

### 6-1. TTP란?

**Test-time Prompting** = 추론 시점에 **프롬프트를 강화**하여 품질을 안정화하는 전략

핵심: **모델을 재학습하지 않고**, 프롬프트 엔지니어링만으로 양자화 품질 저하를 보완한다.

| 전략 | 방법 | 효과 |
|------|------|------|
| **TTP-A** | 출력 규칙/제약을 명시 | 불명확한 입력에서도 방향성 제시 |
| **TTP-B** | few-shot 예시 포함 | 원하는 출력 형식을 "시연"으로 학습 |

### 6-2. TTP가 효과적인 원리

양자화된 모델은 정밀도 손실로 **"추론 여유(inference headroom)"가 줄어든 상태**이다.

"추론 여유"란 모델이 불명확한 입력을 받았을 때 여러 해석 중에서
올바른 것을 골라내는 능력이다. FP16은 이 여유가 충분하지만,
INT4는 정밀도 손실로 인해 여유가 줄어 잘못된 방향을 선택할 확률이 높아진다.

```
불명확한 입력: "그거 얼마나 걸려?"

FP16: 여러 해석 후보 중 맥락에 맞는 것 선택 (추론 여유 ☀️)
INT4: 잘못된 해석 선택 가능 (추론 여유 부족 🌧️)

TTP 적용: "교통 관련 질문입니다. 소요 시간을 답하세요."
INT4 + TTP: 명확한 방향이 제시되어 올바른 추론 유도 (☀️ 복구)
```

### 6-3. TTP 전략별 동작 원리

**TTP-A (규칙 명시)**

"오타가 있어도 의도를 파악하세요", "간결하게 핵심만 답하세요" 같은
**명시적 규칙**을 프롬프트에 포함시킨다.
모델이 어떤 방향으로 추론해야 하는지 **나침반**을 제공하는 효과가 있다.

**TTP-B (Few-shot 예시)**

"Q: 서울→대전 KTX? A: 약 50분~1시간" 같은 **예시 1~2개**를 프롬프트에 포함시킨다.
모델은 예시를 보고 "이런 형식으로 답하면 되는구나"를 즉석에서 학습한다.
출력 형식의 일관성을 높이는 데 특히 효과적이다.

### 6-4. TTP-A와 TTP-B는 역할이 다르다

두 전략은 **보완하는 영역이 다르다.** 실습 결과를 관찰할 때 이 차이를 인식해야 한다.

| 전략 | 핵심 지시 | 오타 교정 | 출력 형식 통일 | 비유 |
|------|---------|:---:|:---:|------|
| **TTP-A** | "오타가 있어도 **의도를 파악**하세요" | ✅ 강함 | △ 약함 | **나침반** (방향 제시) |
| **TTP-B** | "이 **예시와 동일한 형식**으로 답하세요" | △ 약함 | ✅ 강함 | **샘플 답안** (형식 제시) |

예를 들어, 입력에 오타("서울에셔")가 포함된 경우:
- **TTP-A**: "오타가 있어도 의도를 파악하세요"라는 규칙이 있으므로 → "서울에서"로 교정하여 답변
- **TTP-B**: 오타 교정 규칙이 없으므로 → "서울에셔"를 그대로 가져다 쓸 수 있음 (출력 형식은 통일됨)

> **💡 실무에서는 TTP-A + TTP-B를 결합하여 사용한다**
>
> 규칙 명시(A)와 few-shot 예시(B)를 하나의 템플릿에 함께 포함시키면
> **오타 교정 + 출력 형식 통일**을 동시에 달성할 수 있다.

### 6-5. 효과적인 TTP 템플릿 설계 원칙

| 원칙 | Bad 예시 | Good 예시 |
|------|---------|----------|
| **규칙 명시** | "질문에 답해주세요" | "오타가 있어도 의도를 파악하세요. 간결하게 핵심만 답하세요." |
| **출력 형식 지정** | "시간을 알려주세요" | "소요 시간을 '약 X시간 Y분' 형식으로 답하세요" |
| **Few-shot 예시** | (없음) | "예시: Q: 서울→대전 KTX? A: 약 50분~1시간" |

<br>

> **💡 TTP의 한계**
>
> TTP는 프롬프트 길이가 늘어나므로 **입력 토큰 수가 증가**한다.
> 이는 추론 비용 증가로 이어질 수 있다.
> 따라서 "메모리 절감 + 약간의 토큰 증가"와 "품질 유지" 사이의
> 트레이드오프를 고려하여 적용해야 한다.


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: TTP(Test-time Prompting) 템플릿 + 효과 비교
#
# [TTP = 프롬프트 강화로 양자화 품질 저하를 보완하는 전략]
# 모델을 재학습하지 않고, 입력 프롬프트에 "규칙"이나 "예시"를 추가하여
# 양자화 모델의 "추론 여유" 부족을 보완한다.
#
# [TTP-A: 규칙 명시]
# "오타가 있어도 의도를 파악하세요" 같은 명시적 규칙을 포함.
# 모델에게 "이 방향으로 추론하라"는 나침반을 제공하는 효과.
#
# [TTP-B: Few-shot 예시]
# "Q: 서울→대전? A: 약 50분" 같은 예시를 1~2개 포함.
# 모델이 예시를 보고 "이런 형식으로 답하면 되는구나"를 즉석 학습.
# 출력 형식의 일관성을 높이는 데 특히 효과적.
#
# [관찰 포인트]
# - TTP 없음: 오타/모호함에서 품질 저하 발생
# - TTP-A: 규칙 명시로 방향성이 잡히는지 확인
# - TTP-B: 예시로 출력 형식이 통일되는지 확인
# ═══════════════════════════════════════════════════════════

# ========== TTP 템플릿 정의 ==========

# TTP-A: 출력 규칙/제약 강화
# → {question}에 사용자 질문이 삽입된다 (str.format() 사용)
TTP_A = '''다음 질문에 답해주세요.

**규칙:**
1. 오타나 불명확한 표현이 있어도 의도를 파악하세요.
2. 교통 관련 질문은 대략적인 소요 시간(시간 단위)으로 답하세요.
3. 맥락이 부족하면 일반적인 상황을 가정하여 답하세요.
4. 간결하게 핵심만 답하세요.

**질문:** {question}

**답변:**'''

# TTP-B: few-shot 예시 포함
# → 예시 1개만으로도 출력 형식 통일에 효과적
TTP_B = '''다음은 교통 관련 질문과 답변의 예시입니다.

**예시:**
Q: 서울역에서 대전역까지 KTX로 얼마나 걸려요?
A: 서울역에서 대전역까지 KTX로 약 50분~1시간 정도 소요됩니다.

이제 아래 질문에 동일한 형식으로 답해주세요.

**질문:** {question}

**답변:**'''

# ========== 환경 변화 입력에 TTP 적용 ==========
ttp_test = [
    {'type': '오타', 'prompt': '서울에셔 부산까지 KTX로 얼마나 걸리나요?'},
    {'type': '구어체', 'prompt': 'ktx 서울 부산 몇시간? 걍 대충 알려줘 ㅋㅋ'},
    # 조건 판단: 산술 없이 상황 판단이 필요한 질문 → TTP 효과가 잘 드러남
    {'type': '조건판단', 'prompt': '주말에 서울역에서 KTX 자유석 타면 앉을 수 있을까요?'},
    # 산술 추론: 시간 계산이 필요한 질문
    # ※ TTP로도 산술 오류는 교정이 어렵다.
    #    TTP는 "추론 방향 제시"이지 "계산 능력 보완"이 아니기 때문이다.
    #    이 결과를 통해 TTP의 한계를 함께 관찰할 수 있다.
    {'type': '산술추론', 'prompt': '오후 3시에 서울역에서 KTX 타면 부산에 몇 시에 도착해?'},
]

print('TTP 적용 효과 비교')
print('=' * 70)

for sample in ttp_test:
    print(f'\n[{sample["type"]}] 입력: {sample["prompt"]}')
    print('-' * 50)

    # TTP 없음: 원본 프롬프트 그대로
    r0, _, _ = measure_inference(model_int4, tokenizer, sample['prompt'], max_new_tokens=80)
    print(f'TTP 없음: {r0[:150]}')

    # TTP-A: 규칙 명시 → format()으로 {question}에 질문을 삽입
    r1, _, _ = measure_inference(model_int4, tokenizer, TTP_A.format(question=sample['prompt']), max_new_tokens=80)
    print(f'TTP-A:    {r1[:150]}')

    # TTP-B: few-shot 예시 포함
    r2, _, _ = measure_inference(model_int4, tokenizer, TTP_B.format(question=sample['prompt']), max_new_tokens=80)
    print(f'TTP-B:    {r2[:150]}')

# [결과 해석]
# TTP 없음: 오타에서 엉뚱한 답, 모호함에서 맥락 추론 실패
# TTP-A: 규칙이 명시되어 "교통 질문"이라는 방향성이 잡힘
# TTP-B: 예시가 있어 출력 형식까지 통일됨 (가장 안정적)
# → 재학습 없이 프롬프트만으로 품질이 회복되는 것이 TTP의 핵심 가치
print('\n' + '=' * 70)
print('→ TTP를 적용하면 오타/모호한 입력에서도 응답 품질이 안정화된다.')
print('  TTP-B(few-shot)는 출력 형식까지 통일시켜 가장 일관된 결과를 만든다.')
print('  핵심: 모델 재학습 없이 프롬프트 엔지니어링만으로 품질을 복구할 수 있다.')
print()
print('📋 비교 관찰 포인트:')
print('  □ 중국어/일본어 혼입이 사라졌는가?')
print('  □ 존재하지 않는 정보 생성이 줄었는가?')
print('  □ 답변 형식이 일관되게 통일되었는가? (특히 TTP-B)')
print('  □ 구어체/줄임말도 올바르게 해석하는가?')
print('  □ 조건판단(자유석): TTP 적용 후 답변이 더 구체적인가?')
print()
print('⚠️ TTP의 한계:')
print('  산술추론(몇 시 도착?) 결과를 보면, TTP를 적용해도 시간 계산이 부정확할 수 있다.')
print('  TTP는 "추론 방향 제시"이지 "계산 능력 보완"이 아니기 때문이다.')
print('  산술 정확도가 중요한 서비스에서는 양자화 수준을 보수적으로 선택하거나,')
print('  외부 계산 도구(Tool)를 연결하는 Agent 방식이 필요하다. (4-2에서 배운 내용)')


---

## 7. 정리

### 오늘 배운 전체 흐름

```
① LLM은 크다 → GPU 메모리 부족 문제 (챕터 1)
② 비트 수를 줄여 메모리 절감 — FP32 → FP16 → INT4 (챕터 2)
③ FP16 기준선 측정 — 메모리, 속도, 응답 품질 (챕터 3)
④ INT4 양자화 — bitsandbytes + NF4로 메모리 절감 확인 (챕터 4)
⑤ 양자화의 부작용 — 환경 변화 입력에서 품질 저하 관찰 (챕터 5)
⑥ TTP로 복구 — 프롬프트 강화로 품질 안정화 (챕터 6)
```

### 핵심 개념 요약

| 개념 | 한 줄 정리 |
|------|----------|
| **Quantization** | 가중치의 비트 수를 줄여 메모리 절감 (FP16 → INT4) |
| **PTQ** | 학습 완료 후 양자화. 재학습 불필요. 가장 간편한 방식 |
| **NF4** | 정규분포에 최적화된 4비트 양자화. LLM에 가장 적합 |
| **BitsAndBytesConfig** | HuggingFace 연동 양자화 설정. `load_in_4bit=True`가 핵심 |
| **환경 변화** | 오타/노이즈/모호함 등 학습 데이터와 다른 입력. 양자화 모델이 더 취약 |
| **TTP** | 추론 시 프롬프트 강화로 품질 복구. 재학습 불필요 |

### 다음 시간 (5-1) 미리보기

| 구분 | 5-2 (오늘) | 5-1 (다음) |
|------|:---:|:---:|
| 주제 | 추론 최적화 | 학습 최적화 |
| 핵심 | PTQ (양자화 → 메모리 절감) | QLoRA (양자화 + LoRA → 효율적 학습) |
| 결과 | 가벼운 추론 모델 | 도메인 특화 모델 |
| 공통점 | 둘 다 **4비트 NF4** 양자화 사용 |

<br>

> 오늘 배운 INT4/NF4 양자화가 5-1의 QLoRA에서 그대로 사용된다.
> "이미 양자화된 모델 위에 소수의 파라미터(LoRA)만 학습시킨다"는 것이 QLoRA의 핵심이다.

### 자기주도 실습 안내

이제 `실습_5-2_Quantization.ipynb`를 열고,
오늘 배운 개념을 TODO 코드로 직접 구현해 보자.

---

### **Content License Agreement**

<font color='red'><b>**WARNING**</b></font> : 본 자료는 삼성청년SW·AI아카데미의 컨텐츠 자산으로, 보안서약서에 의거하여 어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다.
